# Notebook for Random Forest Regression and SHAPLY calculation

#### Imports

In [1]:
import pandas as pd
import importlib
from RandomForestRegression import random_forest_machine_learning
import question_filtering
importlib.reload(random_forest_machine_learning)
importlib.reload(question_filtering)

<module 'question_filtering' from 'C:\\Users\\Minh\\Desktop\\Psycho Arbeit\\Psycho_DataspellProjekt\\question_filtering.py'>

#### Machine Learning Variables

In [2]:
# ===========================
# Global Notebook Settings
# ===========================
my_target_column = 'How happy are you?'
my_test_size = 0.2
my_random_state = 42

# Model-specific hyperparameters (baseline defaults)
my_number_of_estimators = 100

# SHAP settings
my_number_of_shaply_rows = 5000
my_number_header_rows = 150

##### Model 1A

In [3]:
# Load data 
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')

model_1A, shap_results_1A = random_forest_machine_learning.run_pipeline(data, my_target_column, my_test_size, my_random_state, my_number_of_estimators, my_number_of_shaply_rows, my_number_header_rows)

✅ RMSE: 1.2683
✅ R²: 0.4371
🔍 Starting SHAP  for 5000 rows...


100%|██████████| 1225/1225 [01:24<00:00, 14.42it/s]


Top SHAP features:
                                          Questions     Value  Direction
119  How satisfied with present standard of living?  0.462120  -0.001629
121                 How satisfied with family life?  0.363217  -0.017933
123                 How satisfied with social life?  0.162318   0.006216
122                      How satisfied with health?  0.099711   0.009026
114                I am optimistic about the future  0.074726   0.008199
..                                              ...       ...        ...
152                              Country_Luxembourg  0.000764   0.000284
153                       Country_Macedonia (FYROM)  0.000753  -0.000111
145                                 Country_Hungary  0.000744  -0.000289
160                                  Country_Serbia  0.000691  -0.000019
146                                 Country_Iceland  0.000615  -0.000101

[150 rows x 3 columns]


In [4]:
# Save results
shap_results_1A.to_csv("Results/shap_results_1A.csv", index=False)

##### Model 1A Hyperparameter optimized

In [3]:
# Run Hyperparameter Optimization
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')

model_1A_optimized, shap_results_1A_optimized = random_forest_machine_learning.run_rf_gridsearch_and_shap(data, my_target_column, my_test_size, my_random_state)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

 Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
 Test RMSE: 1.2917
 Test R²: 0.4162
Starting SHAP  for 1000 rows...


 46%|████▌     | 455/1000 [01:49<02:11,  4.15it/s]


KeyboardInterrupt: 

In [9]:
# Save the results
shap_results_1A_optimized.to_csv("Results/shap_results_1A_optimized.csv", index=False)

##### Model 1B - No 'How satisfied with...' questions

In [ ]:
# Load data 
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')
my_target_column = 'How happy are you?'
print("Shape before: ", data.shape)

# Drop 'How satisfied with...' columns
# They have too much predictive value and there are similar questions 
# e.g. "Health condition", "Quality of education system?", "Feel left out of sociey?" 
columns_to_remove = question_filtering.columns_to_remove

data.drop(columns=columns_to_remove, inplace=True, errors='ignore')
print("Shape after: ", data.shape)

model_1B, shap_results_1B = random_forest_machine_learning.run_pipeline(data, my_target_column, my_test_size, my_random_state, my_number_of_estimators, my_number_of_shaply_rows, my_number_header_rows)

In [4]:
# Save the results
shap_results_1B.to_csv("Results/shap_results_1B_iterativ.csv", index=False)

##### Model 1B - No 'How satisfied with...' questions - Hyperparameter optimized

In [4]:
# Load data 
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')
my_target_column = 'How happy are you?'
print("Shape before: ", data.shape)

# Drop 'How satisfied with...' columns
# They have too much predictive value and there are similar questions 
# e.g. "Health condition", "Quality of education system?", "Feel left out of sociey?" 
columns_to_remove = question_filtering.columns_to_remove

data.drop(columns=columns_to_remove, inplace=True, errors='ignore')
print("Shape after: ", data.shape)

model_1B_optimized, shap_results_1B_optimized = random_forest_machine_learning.run_rf_gridsearch_and_shap(data, my_target_column, my_test_size, my_random_state)

Shape before:  (6122, 168)
Shape after:  (6122, 161)
Fitting 3 folds for each of 18 candidates, totalling 54 fits

 Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
 Test RMSE: 1.4206
 Test R²: 0.2938
Starting SHAP  for 1000 rows...


 82%|████████▏ | 815/1000 [03:45<00:51,  3.61it/s]


KeyboardInterrupt: 

In [13]:
# Save the results
shap_results_1B_optimized.to_csv("Results/shap_results_1B_optimized.csv", index=False)

##### Model 2A - Removing features with SHAP < 0.01

In [17]:
# Read previous SHAP results and filter all that are <0.1
shaps = pd.read_csv('Results/shap_results_1B.csv')
low_shap_features = shaps[shaps['Value'] <0.01]
print(low_shap_features)

# Read training data and remove features that have a low SHAP value <0.01
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')
print("Data shape before: ", data.shape)
data.drop(columns=low_shap_features.Questions, inplace=True, errors='ignore')
print("Data shape after: ", data.shape)

model_2A, shap_results_2A = random_forest_machine_learning.run_pipeline(data, my_target_column, my_test_size, my_random_state, my_number_of_estimators, my_number_of_shaply_rows, my_number_header_rows)

                                             Questions     Value  Direction
41       Feel close to people in the area where I live  0.009729  -0.000151
42   Household financial expectations for th 12 mon...  0.009554  -0.000685
43         Likelihood of leaving accom within 6 months  0.009468  -0.000226
44   A person to get support from to raise emergenc...  0.009444   0.000813
45                  No. of problems with accommodation  0.009416   0.001262
..                                                 ...       ...        ...
156                                  Country_Lithuania  0.000333   0.000123
157                                     Country_Latvia  0.000329   0.000209
158                                Country_Netherlands  0.000275   0.000105
159                                   Country_Bulgaria  0.000202  -0.000020
160         Long term care used in 12 months - refusal  0.000005  -0.000005

[120 rows x 3 columns]
Data shape before:  (6122, 168)
Data shape after:  (6122, 48)
✅ 

100%|██████████| 1225/1225 [01:22<00:00, 14.81it/s]


Top SHAP features:
                                            Questions     Value  Direction
39     How satisfied with present standard of living?  0.451144  -0.001027
41                    How satisfied with family life?  0.362159  -0.016643
43                    How satisfied with social life?  0.165451   0.006413
42                         How satisfied with health?  0.102494   0.008250
34                   I am optimistic about the future  0.084387   0.008233
5                                    Health condition  0.067761   0.000559
35  I generally feel that what I do in life is wor...  0.058758  -0.000825
36     I feel I am free to decide how to live my life  0.043386  -0.002928
21                        Can most people be trusted?  0.034017   0.000699
27  Can't find the way because life has become so ...  0.032201  -0.000199
33  Deprivation index: No. of items hhold can't af...  0.026114  -0.000828
40                  How satisfied with accommodation?  0.025658   0.001129
24   

In [18]:
# Save the results
shap_results_2A.to_csv("Results/shap_results_2A.csv", index=False)

#### Model 2A - Removing features with SHAP < 0.01 AND Hyperparameter optimized

In [19]:
# Read previous SHAP results and filter all that are <0.1
shaps = pd.read_csv('Results/shap_results_1B.csv')
low_shap_features = shaps[shaps['Value'] <0.01]
print(low_shap_features)

# Read training data and remove features that have a low SHAP value <0.01
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')
print("Data shape before: ", data.shape)
data.drop(columns=low_shap_features.Questions, inplace=True, errors='ignore')
print("Data shape after: ", data.shape)

model_2A_optimized, shap_results_2A_optimized = random_forest_machine_learning.run_rf_gridsearch_and_shap(data, my_target_column, my_test_size, my_random_state)

                                             Questions     Value  Direction
41       Feel close to people in the area where I live  0.009729  -0.000151
42   Household financial expectations for th 12 mon...  0.009554  -0.000685
43         Likelihood of leaving accom within 6 months  0.009468  -0.000226
44   A person to get support from to raise emergenc...  0.009444   0.000813
45                  No. of problems with accommodation  0.009416   0.001262
..                                                 ...       ...        ...
156                                  Country_Lithuania  0.000333   0.000123
157                                     Country_Latvia  0.000329   0.000209
158                                Country_Netherlands  0.000275   0.000105
159                                   Country_Bulgaria  0.000202  -0.000020
160         Long term care used in 12 months - refusal  0.000005  -0.000005

[120 rows x 3 columns]
Data shape before:  (6122, 168)
Data shape after:  (6122, 48)
Fi

100%|██████████| 1000/1000 [03:26<00:00,  4.84it/s]


Top SHAP features:
                                            Questions     Value  Direction
41                    How satisfied with family life?  0.230346  -0.002920
39     How satisfied with present standard of living?  0.222898  -0.012693
43                    How satisfied with social life?  0.166845  -0.006091
42                         How satisfied with health?  0.129005   0.006452
40                  How satisfied with accommodation?  0.098796  -0.002178
34                   I am optimistic about the future  0.063182   0.003627
35  I generally feel that what I do in life is wor...  0.062855   0.000875
27  Can't find the way because life has become so ...  0.061854  -0.001881
5                                    Health condition  0.058006   0.002995
36     I feel I am free to decide how to live my life  0.054250  -0.004104
31                  Household able to make ends meet?  0.052604   0.002181
33  Deprivation index: No. of items hhold can't af...  0.046205   0.002784
15   

In [20]:
# Save the results
shap_results_2A_optimized.to_csv("Results/shap_results_2A_optimized.csv", index=False)

##### Model 2B - Removing features with SHAP < 0.01 and removing 'How satisfied...' questions

In [ ]:
# Read previous SHAP results and filter all that are <0.1
shaps = pd.read_csv('Results/shap_results_1B.csv')
low_shap_features = shaps[shaps['Value'] <0.01]
print(low_shap_features)

# Read training data and remove features that have a low SHAP value <0.01
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')
print("Data shape before: ", data.shape)
data.drop(columns=low_shap_features.Questions, inplace=True, errors='ignore')
print("Data shape after: ", data.shape)

# Drop 'How satisfied with...' columns
columns_to_remove = question_filtering.columns_to_remove

data.drop(columns=columns_to_remove, inplace=True, errors='ignore')
print("Shape after: ", data.shape)

model_2B, shap_results_2B = random_forest_machine_learning.run_pipeline(data, my_target_column, my_test_size, my_random_state, my_number_of_estimators, my_number_of_shaply_rows, my_number_header_rows)

In [22]:
# Save the results
shap_results_2B.to_csv("Results/shap_results_2B.csv", index=False)

##### Model 2B - Removing features with SHAP < 0.01 and removing 'How satisfied...' questions
##### Hpyerparameter optimized

In [23]:
# Read previous SHAP results and filter all that are <0.1
shaps = pd.read_csv('Results/shap_results_1B.csv')
low_shap_features = shaps[shaps['Value'] <0.01]
print(low_shap_features)

# Read training data and remove features that have a low SHAP value <0.01
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')
print("Data shape before: ", data.shape)
data.drop(columns=low_shap_features.Questions, inplace=True, errors='ignore')
print("Data shape after: ", data.shape)

# Drop 'How satisfied with...' columns
columns_to_remove = question_filtering.columns_to_remove

data.drop(columns=columns_to_remove, inplace=True, errors='ignore')
print("Shape after: ", data.shape)

model_2B_optimized, shap_results_2B_optimized = random_forest_machine_learning.run_rf_gridsearch_and_shap(data, my_target_column, my_test_size, my_random_state)

                                             Questions     Value  Direction
41       Feel close to people in the area where I live  0.009729  -0.000151
42   Household financial expectations for th 12 mon...  0.009554  -0.000685
43         Likelihood of leaving accom within 6 months  0.009468  -0.000226
44   A person to get support from to raise emergenc...  0.009444   0.000813
45                  No. of problems with accommodation  0.009416   0.001262
..                                                 ...       ...        ...
156                                  Country_Lithuania  0.000333   0.000123
157                                     Country_Latvia  0.000329   0.000209
158                                Country_Netherlands  0.000275   0.000105
159                                   Country_Bulgaria  0.000202  -0.000020
160         Long term care used in 12 months - refusal  0.000005  -0.000005

[120 rows x 3 columns]
Data shape before:  (6122, 168)
Data shape after:  (6122, 48)
Sh

100%|██████████| 1000/1000 [03:59<00:00,  4.18it/s]


Top SHAP features:
                                            Questions     Value  Direction
5                                    Health condition  0.124639   0.006821
27  Can't find the way because life has become so ...  0.119135  -0.004247
33  Deprivation index: No. of items hhold can't af...  0.116808   0.002467
38                      How satisfied with education?  0.109077  -0.004888
36     I feel I am free to decide how to live my life  0.104627  -0.004176
35  I generally feel that what I do in life is wor...  0.101601  -0.000912
31                  Household able to make ends meet?  0.089846   0.001166
34                   I am optimistic about the future  0.087103   0.003467
15                       Quality of education system?  0.085214   0.000702
30                       Personal financial situation  0.068956  -0.000169
24                         How much trust the police?  0.060777  -0.006309
28  The value of what I do is not recognised by ot...  0.057019  -0.002834
14   

In [24]:
# Save the results
shap_results_2B_optimized.to_csv("Results/shap_results_2B_optimized.csv", index=False)